5.0.1 Drive i putanje

Montira se Drive i definišu se putanje ka curated podacima i izlazu curated_v2. Ovaj notebook kreira novu podelu samo za LC25000 i RM1000.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, time, math, hashlib
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path("/content/drive/MyDrive/Diplomski")
CURATED = ROOT / "Data" / "curated"
CURATED_V2 = ROOT / "Data" / "curated_v2"
CURATED_V2.mkdir(parents=True, exist_ok=True)

RUNS = ROOT / "Runs"
RUNS.mkdir(parents=True, exist_ok=True)

run_id = time.strftime("%Y%m%d_%H%M%S")
OUT = RUNS / f"leakfree_{run_id}"
OUT.mkdir(parents=True, exist_ok=True)

print("CURATED:", CURATED)
print("CURATED_V2:", CURATED_V2)
print("OUT:", OUT)

Mounted at /content/drive
CURATED: /content/drive/MyDrive/Diplomski/Data/curated
CURATED_V2: /content/drive/MyDrive/Diplomski/Data/curated_v2
OUT: /content/drive/MyDrive/Diplomski/Runs/leakfree_20260316_203053


5.0.2 Helperi: meta, kolone, skupljanje path-ova

Učitava se meta.json i split CSV i nalazi se path/label kolona. Zatim skupljamo sve putanje iz train+val+test u jedinstvenu listu.

In [2]:
def load_meta(ds, root=CURATED):
    with open(root / ds / "meta.json", "r", encoding="utf-8") as f:
        return json.load(f)

def load_split(ds, split, root=CURATED):
    return pd.read_csv(root / ds / f"{split}.csv")

def label_col(meta, df):
    c = meta.get("label_col")
    if c and c in df.columns:
        return c
    for cand in ["label", "target", "y", "class", "Recurred"]:
        if cand in df.columns:
            return cand
    return df.columns[-1]

def path_col(meta, df):
    c = meta.get("path_col")
    if c and c in df.columns:
        return c
    for cand in ["path", "filepath", "image_path", "img_path", "file"]:
        if cand in df.columns:
            return cand
    return None

def gather_all_rows(ds):
    meta = load_meta(ds)
    df_tr = load_split(ds, "train")
    pcol = path_col(meta, df_tr)
    ycol = label_col(meta, df_tr)

    frames = []
    for sp in ["train","val","test"]:
        df = load_split(ds, sp)
        df = df[[pcol, ycol]].copy()
        df["split_orig"] = sp
        frames.append(df)

    all_df = pd.concat(frames, ignore_index=True)
    all_df.rename(columns={pcol: "path", ycol: "label"}, inplace=True)
    all_df["path"] = all_df["path"].astype(str)
    all_df["label"] = all_df["label"].astype(str)
    return meta, all_df

5.1 Brzo hashovanje sa kešom (paralelno)

Ovo je ključ optimizacije: ako već postoji md5_manifest.csv u OUT folderu, koristi se. Ako ne, računa se paralelno i snimi se, pa sledeći put ne radiš ispočetka.

In [3]:
from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm.auto import tqdm

def md5_file(path, chunk=1024*1024):
    h = hashlib.md5()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

def build_md5_manifest(paths, out_csv, workers=4):
    paths = list(dict.fromkeys(paths))
    results = []

    with ProcessPoolExecutor(max_workers=workers) as ex:
        futs = {ex.submit(md5_file, p): p for p in paths}
        for fut in tqdm(as_completed(futs), total=len(futs), desc="md5"):
            p = futs[fut]
            try:
                h = fut.result()
                results.append((p, h))
            except:
                results.append((p, None))

    df = pd.DataFrame(results, columns=["path","md5"]).dropna()
    df.to_csv(out_csv, index=False)
    return df

def get_md5_manifest(ds, all_paths, workers=4):
    out_csv = OUT / f"{ds}_md5_manifest.csv"
    if out_csv.exists():
        df = pd.read_csv(out_csv)
        return df
    df = build_md5_manifest(all_paths, out_csv, workers=workers)
    return df

5.1 Brzo hashovanje sa kešom (paralelno)

Ovo je ključ optimizacije: ako već postoji md5_manifest.csv u OUT folderu, koristi se. Ako ne, računa se paralelno i snimi se, pa sledeći put ne radiš ispočetka.

In [4]:
def group_stratified_split(df, train_ratio=0.7, val_ratio=0.15, test_ratio=0.15, seed=42):
    assert abs(train_ratio + val_ratio + test_ratio - 1.0) < 1e-9

    df = df.copy()
    groups = df.groupby("md5").agg(
        n=("md5","size"),
        label=("label", lambda x: x.value_counts().index[0])
    ).reset_index()

    rng = np.random.default_rng(seed)

    train_groups, val_groups, test_groups = [], [], []

    for label, g in groups.groupby("label"):
        ids = g["md5"].to_numpy()
        rng.shuffle(ids)
        n = len(ids)
        n_tr = int(round(n * train_ratio))
        n_va = int(round(n * val_ratio))
        tr = ids[:n_tr]
        va = ids[n_tr:n_tr+n_va]
        te = ids[n_tr+n_va:]
        train_groups.extend(tr.tolist())
        val_groups.extend(va.tolist())
        test_groups.extend(te.tolist())

    train_set = set(train_groups)
    val_set = set(val_groups)
    test_set = set(test_groups)

    df["split_v2"] = np.where(df["md5"].isin(train_set), "train",
                      np.where(df["md5"].isin(val_set), "val", "test"))
    return df

5.3 Resplit za LC25000 i RM1000 + export u curated_v2

Ovo pravi nove train/val/test CSV fajlove koji su leakage-free po md5. Meta.json se kopira, a u meta možeš dodati oznaku da je v2 i da je group split.

In [5]:
import shutil

def export_v2(ds, df_v2, meta):
    out_dir = CURATED_V2 / ds
    out_dir.mkdir(parents=True, exist_ok=True)

    meta2 = dict(meta)
    meta2["split_version"] = "v2_leakage_free_md5"
    meta2["split_seed"] = 42
    meta2["split_ratio"] = {"train": 0.7, "val": 0.15, "test": 0.15}

    with open(out_dir / "meta.json", "w", encoding="utf-8") as f:
        json.dump(meta2, f, ensure_ascii=False, indent=2)

    for sp in ["train","val","test"]:
        part = df_v2[df_v2["split_v2"] == sp][["path","label"]].copy()
        part.to_csv(out_dir / f"{sp}.csv", index=False)

    return out_dir

def run_resplit(ds, workers=4, seed=42):
    meta, all_df = gather_all_rows(ds)
    all_paths = all_df["path"].tolist()

    md5_df = get_md5_manifest(ds, all_paths, workers=workers)

    df = all_df.merge(md5_df, on="path", how="inner")
    before = len(all_df)
    after = len(df)

    df_v2 = group_stratified_split(df, seed=seed)

    out_dir = export_v2(ds, df_v2, meta)

    summary = {
        "dataset": ds,
        "rows_before": int(before),
        "rows_hashed": int(after),
        "unique_md5": int(df["md5"].nunique()),
        "train": int((df_v2["split_v2"]=="train").sum()),
        "val": int((df_v2["split_v2"]=="val").sum()),
        "test": int((df_v2["split_v2"]=="test").sum()),
    }
    return summary, out_dir

summaries = []
for ds in ["lc25000", "rm1000_lung_history"]:
    s, out_dir = run_resplit(ds, workers=4, seed=42)
    summaries.append(s)
    print("Saved v2:", out_dir)

df_sum = pd.DataFrame(summaries)
df_sum.to_csv(OUT / "v2_split_summary.csv", index=False)
print("Saved:", OUT / "v2_split_summary.csv")
df_sum

md5:   0%|          | 0/25000 [00:00<?, ?it/s]

Saved v2: /content/drive/MyDrive/Diplomski/Data/curated_v2/lc25000


md5:   0%|          | 0/15000 [00:00<?, ?it/s]

Saved v2: /content/drive/MyDrive/Diplomski/Data/curated_v2/rm1000_lung_history
Saved: /content/drive/MyDrive/Diplomski/Runs/leakfree_20260316_203053/v2_split_summary.csv


,dataset,rows_before,rows_hashed,unique_md5,train,val,test
0,lc25000,25000,25000,23720,17518,3738,3744
1,rm1000_lung_history,15000,15000,14195,10501,2253,2246


5.3 Resplit za LC25000 i RM1000 + export u curated_v2

Ovo pravi nove train/val/test CSV fajlove koji su leakage-free po md5. Meta.json se kopira, a u meta možeš dodati oznaku da je v2 i da je group split.


In [6]:
def check_no_overlap(ds):
    meta = load_meta(ds, root=CURATED_V2)
    df_tr = load_split(ds, "train", root=CURATED_V2)
    df_te = load_split(ds, "test", root=CURATED_V2)

    pcol = "path"
    tr_paths = df_tr[pcol].astype(str).tolist()
    te_paths = df_te[pcol].astype(str).tolist()

    man = pd.read_csv(OUT / f"{ds}_md5_manifest.csv")
    m = dict(zip(man["path"], man["md5"]))

    tr_md5 = set([m.get(p) for p in tr_paths if m.get(p) is not None])
    te_md5 = set([m.get(p) for p in te_paths if m.get(p) is not None])

    overlap = tr_md5 & te_md5
    return len(overlap)

for ds in ["lc25000", "rm1000_lung_history"]:
    print(ds, "train-test md5 overlap:", check_no_overlap(ds))

lc25000 train-test md5 overlap: 0
rm1000_lung_history train-test md5 overlap: 0
